# Lab 32: Fine-Tuning Vision AI: Custom YOLO Object Detector

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
The models of Labs 9 to 14 know 80 classes chosen by other people. Your mug, your school bag or the tool of your project is not one of them. In this lab you **teach the Jetson your own objects**. You take pictures of two or three objects, the program **labels** the pictures for you, and you **fine-tune** the small model `yolo26n` so that it finds your objects. This is how real AI products are made: collect data, label it, train, test, improve.

**By the end of this lab you can:**
1. Describe the five steps of making your own detector: collect, label, split, train, test.
2. Collect pictures with a plain background and let OpenCV find the box of the object.
3. Explain the YOLO label format and the folders of a dataset.
4. Train (fine-tune) a model on the Jetson and read precision, recall and mAP50.
5. Test the model in a new situation and explain why it fails or works (overfitting).

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. The live video appears under the cell that starts it. Press **Ctrl + S** often.

> **Part 3, classification of this lab.** *Fine-tuning* a pretrained model on your own data. Lab 31 gave you the words of training (train, validation and test sets, loss, epochs, overfitting); here you use them on a real detector. Compare the result with the baseline of the plain model.
> **Memory.** Training uses most of the 8 GB memory, so only **one student trains at a time** and the others wait (the program tells you). Close the live views of other labs (Stop, then Kernel, Shut Down All Kernels) before you train.
> **Privacy.** Do not collect pictures of faces or people. Use objects.

## 2. Background

**Fine-tuning.** The model `yolo26n.pt` already knows how to see: edges, shapes and textures. We do not train it from nothing. We start from its weights and train for a few minutes on **your** pictures so that it learns your classes. This is called **fine-tuning** or **transfer learning**. A few dozen pictures per object are enough for a first model.

**A dataset.** A detector needs, for every picture, a **label**: the class and the box of each object. YOLO keeps one text file per picture with one line per object:

`class_number  x_centre  y_centre  width  height`

All four numbers are fractions of the picture (0 to 1). For example `1 0.50 0.40 0.20 0.30` is class 1, in the middle, a bit above the centre, 20 % of the width and 30 % of the height. The folders are:

| Folder | Content |
|---|---|
| `dataset/images/train` and `dataset/labels/train` | Pictures and labels the model **learns** from (about 80 %) |
| `dataset/images/val` and `dataset/labels/val` | Pictures and labels used to **check** the model on pictures it has not learned (about 20 %) |
| `dataset/data.yaml` | The class names and the folder paths |

**Labelling the easy way.** Drawing boxes by hand takes long. Here is a trick that uses Lab 6: put **one object on a plain background** (a white sheet of paper) with a **fixed camera**. Take a picture of the empty background first. For each new picture, subtract the empty background (`cv2.absdiff`). What remains is the object. The biggest remaining shape is the object and its box is the label. You check the boxes by eye afterwards.

| Step | OpenCV function | What it does |
|---|---|---|
| 1 | `cv2.absdiff` | The difference between the picture and the empty background |
| 2 | `cv2.GaussianBlur` | Removes the noise |
| 3 | `cv2.threshold` | Black and white: white where the picture changed |
| 4 | `cv2.morphologyEx` | Closes holes and removes small spots |
| 5 | `cv2.findContours`, `cv2.boundingRect` | The biggest white shape and its box |

**Good data beats a big model.** A model learns what is in the pictures, nothing else. If every mug picture has a white background, the model may learn that "white background = mug". Vary the **position**, the **turn**, the **distance** and the **light**. Add pictures **without** any object so the model learns when there is nothing. Part D tests this.

**Training numbers.** After the training you get:

| Number | Meaning |
|---|---|
| Precision | Of the objects the model found, how many are right |
| Recall | Of the real objects, how many the model found |
| mAP50 | One score from 0 to 1 that combines both (a box counts as right when it overlaps the real box by at least 50 %). Higher is better. |

**Overfitting.** A model that scores very high on its own pictures but fails on a new place has **memorised** instead of learned. The check pictures (`val`) and the test in a new place (Part D) show it.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(32)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Plan and Prepare (20 min)

### Step 1: Choose Your Objects
Choose **2 or 3** objects. Good objects are **different from each other** in shape and colour, **bigger than a hand-palm finger** (a thin pen is hard at this size) and **not in the 80 classes** (so you really teach something new). Use a plain sheet (white paper) as the background and fix the camera so it does not move.

| Good | Not so good |
|---|---|
| A mug, a stapler, a tape roll, a toy, a small box, a TV remote | Two white cups that look the same, a transparent bottle, a thin pen |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Object 1:
- Object 2:
- Object 3 (or none):
- Why are these objects different enough for a model to tell them apart?:


### Step 2: The Settings
`CLASSES` are the names of your objects (one word, no spaces). `IMAGES_PER_CLASS` is how many pictures you will take of each. `EPOCHS` is how many times the model goes through all pictures when it trains. `IMGSZ` is the picture size for training and testing (smaller is faster).

In [ ]:
# ---------------- Settings you can change ----------------
CLASSES = ["mug", "toy"]  # the names of your objects (one word each)
IMAGES_PER_CLASS = 40     # pictures to take of each object
EPOCHS = 30               # how many times the model sees all the pictures
IMGSZ = 320               # picture size for training and testing
DIFF_LIMIT = 35           # how different from the background a pixel must be (0 to 255)
MIN_AREA = 0.01           # the object must cover at least this share of the picture
MAX_AREA = 0.60           # ... and at most this share
# ----------------------------------------------------------
print("Classes:", CLASSES)

### Step 3: Get the Base Model
The base model is `yolo26n.pt`. The cell uses the copy from the earlier labs.

In [ ]:
import csv
import glob
import json
import os
import random
import shutil
import time

import cv2
import numpy as np
from IPython.display import Image, display

import labview

print("This notebook is saved in:", os.getcwd())
for source in ["../models/yolo26n.pt", "../lab16/yolo26n.pt", "../lab15/yolo26n.pt"]:
    if os.path.exists(source) and not os.path.exists("yolo26n.pt"):
        shutil.copy(source, "yolo26n.pt")
        print("copied", source)
print("Base model:", "found" if os.path.exists("yolo26n.pt") else "will be downloaded when you train (needs the internet)")

---
# Part B: Collect and Label (50 min)

### Step 1: The Labelling Function
Read this cell. `find_box` follows the five steps of the Background section and returns the box `(x, y, w, h)` of the object in pixels, or `None` if it finds nothing (or something that is too small or too big).

In [ ]:
def find_box(frame, background):
    """The box (x, y, w, h) of the object that is new in the picture compared with the empty background, or None."""
    diff = cv2.absdiff(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), cv2.cvtColor(background, cv2.COLOR_BGR2GRAY))
    diff = cv2.GaussianBlur(diff, (7, 7), 0)
    _, mask = cv2.threshold(diff, DIFF_LIMIT, 255, cv2.THRESH_BINARY)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8), iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    biggest = max(contours, key=cv2.contourArea)
    share = cv2.contourArea(biggest) / (frame.shape[0] * frame.shape[1])
    if share < MIN_AREA or share > MAX_AREA:
        return None
    return cv2.boundingRect(biggest)


# a quick check with a fake picture: a white page with a dark rectangle at x 200 to 300 and y 150 to 330
page = np.full((480, 640, 3), 235, np.uint8)
thing = page.copy()
cv2.rectangle(thing, (200, 150), (300, 330), (40, 60, 160), -1)
print("Box found:", find_box(thing, page), "(expected about (200, 150, 101, 181))")
print("Empty page:", find_box(page, page), "(expected None)")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did the check give a box close to the expected one?:
- Why does the function return `None` when the object is too big (`MAX_AREA`)?:


### Step 2: Collect the Pictures
Set up: a plain sheet as the background, the camera fixed, good even light.

1. Run the cell. **Remove all objects** and press **Take background**.
2. Choose a class in the list, put the object on the sheet and watch the **green box**. If the box is good, press **Capture**. Move or turn the object a little and press **Capture** again, until you have `IMAGES_PER_CLASS` pictures of that class. Change the position, the angle (front, side, top), the distance and lay the object down and stand it up.
3. Choose the next class and repeat.
4. Optional: choose `empty` and take pictures of **other places without any object** (point the camera at a desk, a wall). These teach the model when there is nothing to find. The box is not needed for `empty` pictures.

Each picture is saved with its box, in `raw/<class>/`. If the box is wrong or nothing is found, nothing is saved. If the light changes much, take the background again. Press **Stop** when you are done.

In [ ]:
run = {"frame": None, "bg": None, "box": None, "note": "Take the background first (no object on the sheet)"}


def counts():
    return {name: len(glob.glob(f"raw/{name}/*.jpg")) for name in CLASSES + ["empty"]}


def take_background(state):
    if run["frame"] is not None:
        run["bg"] = run["frame"].copy()
        cv2.imwrite("background.jpg", run["bg"])
        run["note"] = "Background saved. Put an object on the sheet."


def capture(state):
    frame, box, name = run["frame"], run["box"], state["Class"]
    if frame is None or run["bg"] is None:
        run["note"] = "Take the background first"
        return
    if name != "empty" and box is None:
        run["note"] = "No box found: nothing saved"
        return
    folder = "raw/" + name
    os.makedirs(folder, exist_ok=True)
    number = len(glob.glob(f"{folder}/*.jpg")) + 1
    stem = f"{folder}/{name}_{number:04d}"
    cv2.imwrite(stem + ".jpg", frame)
    with open(stem + ".box", "w") as f:
        f.write("none" if name == "empty" else " ".join(str(int(v)) for v in box))
    run["note"] = f"Saved {name} number {number}"


def process_collect(frame, state):
    run["frame"] = frame.copy()
    shown = frame.copy()
    run["box"] = find_box(frame, run["bg"]) if run["bg"] is not None else None
    if run["box"] is not None and state["Class"] != "empty":
        x, y, w, h = run["box"]
        cv2.rectangle(shown, (x, y), (x + w, y + h), (0, 255, 0), 2)
    c = counts()
    line = "   ".join(f"{k}: {v}" for k, v in c.items())
    cv2.putText(shown, line, (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
    cv2.putText(shown, run["note"][:70], (10, shown.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    return shown


live = labview.live(process_collect, controls={"Class": CLASSES + ["empty"]},
                    actions={"Take background": take_background, "Capture": capture})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many pictures did you take of each class?:
- What did you change between the pictures (position, angle, light)?:
- When did the box go wrong? What did you do?:


### Step 3: Check the Labels
A model learns the mistakes in its labels, so **look at the pictures**. The cell shows a **contact sheet** of 30 pictures at a time with the box and a number. Pictures with a bad box (too small, too big, the shadow, the wrong thing) must go. Write their numbers in `BAD`.

In [ ]:
def load_items():
    """Every collected picture: (class number, class name, picture file, box or None). The order is always the same."""
    items = []
    for number, name in enumerate(CLASSES):
        for jpg in sorted(glob.glob(f"raw/{name}/*.jpg")):
            box = open(jpg[:-4] + ".box").read().split()
            items.append((number, name, jpg, [int(v) for v in box] if box and box[0] != "none" else None))
    for jpg in sorted(glob.glob("raw/empty/*.jpg")):
        items.append((-1, "empty", jpg, None))
    return items


def show_sheet(start=0, count=30, columns=6):
    """A contact sheet of the pictures start ... start + count with their numbers and boxes."""
    items = load_items()
    tiles = []
    for index in range(start, min(start + count, len(items))):
        number, name, jpg, box = items[index]
        picture = cv2.imread(jpg)
        if box is not None:
            x, y, w, h = box
            cv2.rectangle(picture, (x, y), (x + w, y + h), (0, 255, 0), 4)
        tile = cv2.resize(picture, (213, 160))
        cv2.putText(tile, f"{index} {name}", (4, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        tiles.append(tile)
    if not tiles:
        print("No pictures yet. Collect pictures in Step 2.")
        return
    while len(tiles) % columns:
        tiles.append(np.zeros_like(tiles[0]))
    rows = [np.hstack(tiles[i:i + columns]) for i in range(0, len(tiles), columns)]
    display(Image(data=cv2.imencode(".jpg", np.vstack(rows))[1].tobytes()))
    print(f"Pictures {start} to {min(start + count, len(items)) - 1} of {len(items)}")


show_sheet(0)

Show the next 30 pictures by changing the number (30, 60, ...). Then write the numbers of the bad pictures in `BAD` and run the cell after it.

In [ ]:
show_sheet(30)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many pictures had a bad box?:
- What caused the bad boxes (shadow, reflection, the hand, the object at the edge)?:


### Step 4: Build the Dataset
The cell takes the good pictures, splits each class into **train (80 %)** and **val (20 %)** and writes the pictures, the label files and `data.yaml` in the YOLO format. Change `BAD` first. The split is random but the same every time (`seed`), so you can repeat it.

In [ ]:
BAD = []                 # the numbers of the pictures to leave out, for example [3, 17, 41]


def build_dataset(bad=(), val_share=0.2, seed=1):
    shutil.rmtree("dataset", ignore_errors=True)
    for part in ("train", "val"):
        os.makedirs(f"dataset/images/{part}")
        os.makedirs(f"dataset/labels/{part}")
    groups = {}
    for index, (number, name, jpg, box) in enumerate(load_items()):
        if index not in bad:
            groups.setdefault(name, []).append((number, jpg, box))
    totals = {"train": 0, "val": 0}
    for name, items in groups.items():
        random.Random(seed).shuffle(items)
        n_val = max(1, round(len(items) * val_share)) if len(items) > 1 else 0
        for position, (number, jpg, box) in enumerate(items):
            part = "val" if position < n_val else "train"
            stem = os.path.splitext(os.path.basename(jpg))[0]
            shutil.copy(jpg, f"dataset/images/{part}/{stem}.jpg")
            height, width = cv2.imread(jpg).shape[:2]
            with open(f"dataset/labels/{part}/{stem}.txt", "w") as f:
                if box is not None:                       # a picture of "empty" gets an empty label file
                    x, y, w, h = box
                    f.write(f"{number} {(x + w / 2) / width:.6f} {(y + h / 2) / height:.6f} {w / width:.6f} {h / height:.6f}\n")
            totals[part] += 1
    names = "\n".join(f"  {i}: {name}" for i, name in enumerate(CLASSES))
    with open("dataset/data.yaml", "w") as f:
        f.write(f"path: {os.path.abspath('dataset')}\ntrain: images/train\nval: images/val\nnames:\n{names}\n")
    print("Pictures to learn from (train):", totals["train"])
    print("Pictures to check with (val):  ", totals["val"])
    print("Classes:", CLASSES)


build_dataset(BAD)

Look at one label file and the `data.yaml` to see the format:

In [ ]:
label = sorted(glob.glob("dataset/labels/train/*.txt"))
if label:
    print(label[0])
    print(open(label[0]).read())
print(open("dataset/data.yaml").read())

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Label file shown: which class number is it and where is the object (x_centre, y_centre)?:
- Why are the numbers fractions between 0 and 1 and not pixels?:
- Why do we keep 20 % of the pictures out of the training?:


---
# Part C: Train the Model (35 min)

### Step 1: Train
Run the cell. If another student trains, the program says so and you wait. The training prints one line per epoch. Watch the numbers `box_loss` (it should go **down**) and `mAP50` (it should go **up**). Do not run other heavy programs while it trains. It takes a few minutes. The result is saved in `runs/mine/weights/best.pt`.

In [ ]:
from ultralytics import YOLO

with labview.train_lock():
    started = time.time()
    model = YOLO("yolo26n.pt")
    model.train(data=os.path.abspath("dataset/data.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=8, device=0, workers=2,
                project=os.path.abspath("runs"), name="mine", exist_ok=True, plots=False)
    best = str(model.trainer.best)
print(f"Training took {(time.time() - started) / 60:.1f} minutes. Best model: {best}")

### Step 2: Read the Results
The training saves a table `results.csv`. The cell prints a line every 5 epochs and the last line, with the **loss** (how wrong the model is, lower is better) and the **mAP50**, **precision** and **recall** on the `val` pictures.

In [ ]:
path = os.path.join(os.path.dirname(os.path.dirname(best)), "results.csv")
with open(path, newline="") as f:
    rows = [{k.strip(): v.strip() for k, v in row.items()} for row in csv.DictReader(f)]
def number(row, key):
    return float(row.get(key, "nan"))          # nan when this column does not exist in your version


print(f"{'epoch':>5} {'box_loss':>9} {'precision':>10} {'recall':>8} {'mAP50':>7}")
for i, row in enumerate(rows):
    if i % 5 == 0 or i == len(rows) - 1:
        print(f"{row['epoch']:>5} {number(row, 'train/box_loss'):>9.3f} {number(row, 'metrics/precision(B)'):>10.3f} "
              f"{number(row, 'metrics/recall(B)'):>8.3f} {number(row, 'metrics/mAP50(B)'):>7.3f}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Epochs and training time:
- box_loss at the first and the last epoch:
- Final precision, recall and mAP50:
- Is the model good? (mAP50 above 0.8 is good for this lab):


---
# Part D: Test in the Real World and Improve (45 min)

### Step 1: Run Your Model Live
The program loads `best.pt` and finds your objects. It counts each class on the screen. Test your objects **on the plain sheet** first, then **in a new place**: on a desk, in your hand, with other objects around, in dim light. Press **Stop** when you are done.

In [ ]:
mine = YOLO(best)


def process(frame, state):
    result = mine(frame, imgsz=IMGSZ, conf=state["Confidence"], device=0, verbose=False)[0]
    shown = result.plot()
    names = [result.names[int(c)] for c in result.boxes.cls]
    line = "   ".join(f"{n}: {names.count(n)}" for n in CLASSES)
    cv2.putText(shown, line, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    return shown


live = labview.live(process, controls={"Confidence": (0.05, 0.95, 0.05, 0.5)})

### Step 2: Test Table
Test each situation with your objects and write what happened.

| Test | Situation | Expected | What happened |
|---|---|---|---|
| 1 | The object on the plain sheet, like the training pictures | Found | |
| 2 | The object on a desk with a different colour | Found? | |
| 3 | The object in your hand | Found? | |
| 4 | The object far away (about 1 metre) | Found? | |
| 5 | Dim light or a shadow | Found? | |
| 6 | An object that is **not** one of your classes | Nothing | |
| 7 | An empty desk | Nothing | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1:
- Test 2:
- Test 3:
- Test 4:
- Test 5:
- Test 6 (false detection?):
- Test 7 (false detection?):
- Which tests failed and what does that tell you about your pictures?:


### Step 3: Improve the Data
Most failures are fixed by **better data**, not by more epochs. Choose what went wrong and fix it:

| Problem | Fix |
|---|---|
| Not found on a new background | Take more pictures **on other backgrounds**. The auto labelling needs the plain sheet, so use a coloured paper, a wooden table or a cloth as the new "plain" background: take a **new background picture** each time. |
| Not found far away or small | Take pictures from farther away |
| False detections on empty places | Take **empty** pictures of those places (class `empty`) |
| Two classes are mixed up | Take more pictures of the two, from more angles |

Then collect more pictures (run Step 2 of Part B again, the old pictures are kept), check them (Step 3), build the dataset (Step 4), train again (Part C, you can lower `EPOCHS` to 20) and test again.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What did you improve?:
- mAP50 before and after:
- Did the test table get better? Which tests?:


---
# Part E: Present (30 min)
Fill in the report. Then give a **3-minute demonstration**: show the collection, a contact sheet, the training numbers, the live detection in a new place, and explain one failure and how you fixed it.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your objects and why you chose them:
- Number of pictures per class and what you varied:
- Training: epochs, time, final mAP50, precision, recall:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Where the model worked and where it failed:
- One problem you found and how you fixed it:
- One idea to make the model better in future:


> **Where next.** In Lab 16 you will make your model **fast with TensorRT** (Lab 15) and use it in an application with a zone and an alarm.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The background is taken and the green box follows the object | |
| 2 | At least 30 pictures per class are collected, with different positions and angles | |
| 3 | The contact sheet is checked and the bad pictures are removed with `BAD` | |
| 4 | The dataset is built and the label file and `data.yaml` are explained | |
| 5 | The model is trained and the results table is read (loss, precision, recall, mAP50) | |
| 6 | The live detection finds your objects on the plain sheet | |
| 7 | The test table is complete (7 tests) | |
| 8 | One improvement is made and the mAP50 or the tests are compared | |
| 9 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. What is fine-tuning? Why is it better than training a model from nothing?:
- 2. What are the five numbers of one line in a YOLO label file?:
- 3. How does the program find the box of an object on a plain background? Name three OpenCV functions it uses.:
- 4. Why do we split the pictures into train and val?:
- 5. What do precision and recall mean? Give one example of each in your own project.:
- 6. Your model scores mAP50 = 0.98 but fails on a new desk. What is this problem called, and how do you fix it?:
- 7. Why do we add pictures of `empty` places?:
- 8. Why can only one student train at a time on the Jetson?:
- 9. Which change helps most when the model confuses two classes? (a) Train 10 times longer on the same pictures (b) Take more pictures of both from more angles (c) Make the pictures smaller (d) Rename the classes:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| The green box does not appear | No background taken, or the object is too small, too big or like the background | Press **Take background** with no object on the sheet. Use a plain sheet that differs from the object. Check `MIN_AREA` and `MAX_AREA`. |
| The box covers the shadow or the hand | The light is uneven or something else moves | Light from above, keep your hand out of the picture, raise `DIFF_LIMIT` (for example 45) |
| The box is split or too small | The object has the colour of the sheet | Use another background colour. Lower `DIFF_LIMIT` a little. |
| `No box found: nothing saved` | The object is not seen as new | See the two lines above. Take the background again. |
| `FileNotFoundError ... raw/...` | No pictures of that class were collected | Collect pictures of every class in `CLASSES` (Step 2 of Part B) |
| The training says `waiting` | Another student trains | Wait. Do not stop the cell. |
| `CUDA out of memory` while training | Other notebooks use the memory | Press Stop in all live views, Kernel, Shut Down All Kernels, then run the notebook again from the top |
| `AttributeError ... trainer` or `best` is not defined | The training cell did not finish | Run the training cell again and wait until it prints `Best model` |
| mAP50 is low (below 0.5) | Too few or bad pictures, or too few epochs | Check the contact sheet, take more pictures, raise `EPOCHS` to 50 |
| The model finds your object everywhere | The model learned the background | Add `empty` pictures and pictures on other backgrounds |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(32)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Ultralytics, Train mode: https://docs.ultralytics.com/modes/train/
- Ultralytics, Detection dataset format: https://docs.ultralytics.com/datasets/detect/
- Ultralytics, YOLO performance metrics: https://docs.ultralytics.com/guides/yolo-performance-metrics/
- Roboflow Universe (ready datasets and labelling tools): https://universe.roboflow.com/